# 🛒 Amazon Reviews 2023 - Electronics
### Exploratory Data Analysis (EDA)

We work on a sample of electronics product reviews from the **Amazon Reviews 2023** dataset (McAuley Lab).
We look at the data from two angles:

- **Reviews** (`reviews_clean`) - one opinion = one row (rating, text, time, "helpful" votes).
- **Products** (`meta_clean`) - one product = one row (category, price, average rating).

The goal here is to show **what this dataset is, what it contains and which patterns it reveals** - before
we move on to text embeddings and dimensionality reduction.

In [33]:
import sys
sys.path.append("..")

import re
import math
from collections import Counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import config
from src import storage, preprocessing

# Consistent visual theme for the whole presentation.
NAVY, PLUM, AMBER, RED, GREEN = "#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B8A6E"
RATING_COLORS = {"1": "#C73E1D", "2": "#E8743B", "3": "#F1B211", "4": "#7FB069", "5": "#2E8B57"}

pio.templates["eda"] = go.layout.Template(
    layout=dict(
        font=dict(family="Segoe UI, Arial", size=14, color="#222"),
        title=dict(x=0.02, font=dict(size=20)),
        colorway=[NAVY, PLUM, AMBER, GREEN, RED],
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(l=60, r=40, t=80, b=50),
        height=520,
        xaxis=dict(showgrid=True, gridcolor="#EEE", zeroline=False),
        yaxis=dict(showgrid=True, gridcolor="#EEE", zeroline=False),
    )
)
pio.templates.default = "eda" 

In [34]:
df = storage.load_stage("reviews_clean")
meta = storage.load_stage("meta_clean")

# Helper rating columns as a category (for coloring and axes).
df["rating_int"] = df["rating"].astype(int)
df["rating_str"] = df["rating_int"].astype(str)

print(f"Reviews  : {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Products : {meta.shape[0]:,} rows x {meta.shape[1]} cols")
df.head(3)

Reviews  : 199,954 rows x 13 cols
Products : 5,000 rows x 18 cols


,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,text_len,title_len,rating_int,rating_str
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-07-18 22:58:37.948,0,True,1433,33,3,3
1,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-06-20 18:42:29.731,0,True,225,39,1,1
2,5.0,Excellent!,I love these. They even come with a carry case...,B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2018-04-07 09:23:37.534,0,True,469,10,5,5


## 1. Dataset at a glance

First, the key numbers: how many reviews, products and users we have, and what the average rating looks like.

In [35]:
n_reviews  = len(df)
n_products = df["parent_asin"].nunique()
n_users    = df["user_id"].nunique()
avg_rating = df["rating"].mean()
pct_verif  = 100 * df["verified_purchase"].mean()
years_span = df["timestamp"].dt.year.max() - df["timestamp"].dt.year.min() + 1

cards = [
    ("Reviews",             n_reviews,  ",.0f", "", NAVY),
    ("Products",            n_products, ",.0f", "", PLUM),
    ("Users",               n_users,    ",.0f", "", AMBER),
    ("Average rating",      avg_rating, ".2f",  "", GREEN),
    ("Verified purchases",  pct_verif,  ".0f",  "%", NAVY),
    ("Years covered",       years_span, ".0f",  "", PLUM),
]

fig = make_subplots(rows=2, cols=3, specs=[[{"type": "indicator"}] * 3] * 2,
                    vertical_spacing=0.3, horizontal_spacing=0.05)
for i, (label, value, vfmt, suffix, color) in enumerate(cards):
    fig.add_trace(
        go.Indicator(
            mode="number", value=value,
            number=dict(valueformat=vfmt, suffix=suffix, font=dict(size=44, color=color)),
            title=dict(text=label, font=dict(size=15, color="#666")),
        ),
        row=i // 3 + 1, col=i % 3 + 1,
    )
fig.update_layout(height=360, title="Dataset at a glance")
fig.show()

## 2. Rating distribution - the classic "J" shape

Reviews on Amazon are heavily skewed toward 5★: people are more likely to rate when they are
**very satisfied** or **very unhappy**. Middle ratings appear least often.

In [36]:
rc = df["rating_str"].value_counts().reindex(["1", "2", "3", "4", "5"])
rc_pct = 100 * rc / rc.sum()

fig = go.Figure(go.Bar(
    x=rc.index, y=rc.values,
    marker_color=[RATING_COLORS[r] for r in rc.index],
    text=[f"{v:,}<br>{p:.0f}%" for v, p in zip(rc.values, rc_pct.values)],
    textposition="outside",
))
fig.update_layout(
    title="Distribution of ratings (1-5 stars)",
    xaxis_title="Rating (stars)", yaxis_title="Number of reviews",
    yaxis=dict(range=[0, rc.max() * 1.18]),
)
fig.show()

In [37]:
# Do verified purchases rate differently than unverified ones?
comp = (df.groupby("verified_purchase")["rating_str"]
          .value_counts(normalize=True).rename("share").reset_index())
comp["Purchase"] = comp["verified_purchase"].map({True: "Verified", False: "Unverified"})

fig = px.bar(
    comp, x="Purchase", y="share", color="rating_str",
    category_orders={"rating_str": ["1", "2", "3", "4", "5"]},
    color_discrete_map=RATING_COLORS, text_auto=".0%",
    title="Do verified purchases rate differently?",
    labels={"share": "Share of reviews", "rating_str": "Rating", "Purchase": ""},
)
fig.update_layout(yaxis_tickformat=".0%", legend_title="Rating")
fig.show()

### What does each rating actually sound like?

One representative review per rating (1-5 stars), straight from the data - to make the numbers above concrete.

In [38]:
import html
from IPython.display import HTML

# One representative review per rating: readable length, reproducible pick.
rows = []
for r in [1, 2, 3, 4, 5]:
    pool = df[(df["rating_int"] == r) & (df["text_len"].between(120, 400))]
    pool = pool if not pool.empty else df[df["rating_int"] == r]
    rows.append(pool.sample(1, random_state=7).iloc[0])

cards = []
for row in rows:
    r = int(row["rating_int"])
    color = RATING_COLORS[str(r)]
    stars = "\u2605" * r + "\u2606" * (5 - r)
    title = html.escape(str(row["title"]).strip()) or "(no title)"
    text = html.escape(str(row["text"]).strip())
    if len(text) > 600:
        text = text[:600].rstrip() + "..."
    # overflow-wrap + word-break stop long tokens (product codes, links) from
    # spilling outside the card; box-sizing keeps padding inside the width.
    cards.append(
        f'<div style="border-left:6px solid {color};background:#fafafa;'
        f'padding:10px 16px;margin:8px 0;border-radius:4px;box-sizing:border-box;'
        f'overflow-wrap:break-word;word-break:break-word;">'
        f'<div style="color:{color};font-size:18px;font-weight:700;">{stars} &nbsp; {r} / 5</div>'
        f'<div style="font-weight:600;margin:3px 0;overflow-wrap:break-word;">{title}</div>'
        f'<div style="color:#333;font-size:14px;line-height:1.45;'
        f'overflow-wrap:break-word;word-break:break-word;">{text}</div></div>'
    )

HTML('<div style="font-family:Segoe UI,Arial;max-width:900px;box-sizing:border-box;">'
     + "".join(cards) + "</div>")

## 3. What and how customers write

Review length, its relationship to the rating, and the words that most distinguish positive from
negative opinions.

In [39]:
median_len = df["text_len"].median()
fig = px.histogram(df, x="text_len", nbins=60,
                   title="Distribution of review length (in characters)",
                   labels={"text_len": "Number of characters"})
fig.update_traces(marker_color=NAVY)
fig.add_vline(x=median_len, line_dash="dash", line_color=RED,
              annotation_text=f"median ~ {median_len:.0f} chars",
              annotation_position="top right")
fig.update_layout(yaxis_title="Number of reviews", bargap=0.05)
# Trim the longest 2% so the chart stays readable (the tail reaches ~10,000 chars).
fig.update_xaxes(range=[0, df["text_len"].quantile(0.98)])
fig.show()

In [40]:
fig = px.box(df, x="rating_str", y="text_len", color="rating_str",
             category_orders={"rating_str": ["1", "2", "3", "4", "5"]},
             color_discrete_map=RATING_COLORS,
             title="Do unhappy customers write more? Review length by rating",
             labels={"rating_str": "Rating", "text_len": "Number of characters"})
fig.update_yaxes(range=[0, df["text_len"].quantile(0.95)])
fig.update_layout(showlegend=False)
fig.show()

In [41]:
# Which words most distinguish negative (1-2 star) from positive (5 star) reviews?
# Method: weighted log-odds ratio with an informative Dirichlet prior ("Fightin' Words",
# Monroe et al. 2008). For every word we compare its rate in 1-2 star vs 5 star reviews,
# then divide by the standard error, so rare high-variance words do not dominate.
STOP = set(("the a an and or but if of to in on at for with from by is are was were be been being it its this "
            "that these those i you he she they we my your our their as so not no nor do does did doing have has "
            "had having will would shall should can could may might must just get gets got getting also more most "
            "some any only even very too much many them then than there here what which who whom whose when where "
            "why how all both each few other such own same me him her us up out off over under again once into "
            "about because while after before above below down").split())
# Incentivized-review disclosure boilerplate ("received free / at a discount in exchange
# for my honest, unbiased review") - common in 5 star reviews but not real sentiment.
BOILERPLATE = set(("honest honestly unbiased biased bias exchange discount discounted freely "
                   "complimentary promotional reviewer opinion opinions").split())
EXCLUDE = STOP | BOILERPLATE

def word_counts(texts):
    c = Counter()
    for t in texts:
        for w in re.findall(r"[a-z']+", str(t).lower()):
            w = w.strip("'")
            if len(w) > 2 and w not in EXCLUDE:
                c[w] += 1
    return c

pos = word_counts(df.loc[df["rating"] >= 5, "text"])
neg = word_counts(df.loc[df["rating"] <= 2, "text"])
n_pos, n_neg = sum(pos.values()), sum(neg.values())
total = n_pos + n_neg
vocab = [w for w in set(pos) | set(neg) if pos[w] + neg[w] >= 30]

PRIOR = 1000.0  # total prior mass, spread over words by their background frequency

def z_score(w):  # >0 => typical of 5 star,  <0 => typical of 1-2 star
    a_w = PRIOR * (pos[w] + neg[w]) / total
    delta = (math.log((pos[w] + a_w) / (n_pos + PRIOR - pos[w] - a_w))
             - math.log((neg[w] + a_w) / (n_neg + PRIOR - neg[w] - a_w)))
    return delta / math.sqrt(1.0 / (pos[w] + a_w) + 1.0 / (neg[w] + a_w))

selected = sorted(vocab, key=z_score)[:12] + sorted(vocab, key=z_score, reverse=True)[:12]
selected = sorted(set(selected), key=z_score)
vals = [z_score(w) for w in selected]
colors = [RED if v < 0 else GREEN for v in vals]

fig = go.Figure(go.Bar(x=vals, y=selected, orientation="h", marker_color=colors))
fig.add_vline(x=0, line_color="#888")
fig.update_layout(
    title="Words that set 1-2 star apart from 5 star reviews",
    xaxis_title="More typical of 1-2 star   <---   weighted log-odds (z-score)   --->   more typical of 5 star",
    yaxis_title="Keyword",
    height=640, bargap=0.25,
    margin=dict(l=130),
)
fig.update_yaxes(automargin=True)
fig.show()

## 4. Which reviews are helpful?

"Helpful" votes have a very long tail - most reviews get none, while a few collect hundreds.

In [42]:
hv = df.groupby("rating_int")["helpful_vote"].mean().reindex([1, 2, 3, 4, 5])

fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58],
                    subplot_titles=("Avg. \"helpful\" votes by rating",
                                    "Distribution of \"helpful\" votes (log scale)"))
fig.add_trace(go.Bar(x=hv.index.astype(str), y=hv.values,
                     marker_color=[RATING_COLORS[str(r)] for r in hv.index],
                     showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=df["helpful_vote"], nbinsx=60, marker_color=NAVY,
                           showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="Rating", row=1, col=1)
fig.update_yaxes(title_text="Avg. number of votes", row=1, col=1)
fig.update_xaxes(title_text="\"Helpful\" votes", row=1, col=2)
fig.update_yaxes(type="log", title_text="Number of reviews (log)", row=1, col=2)
fig.update_layout(height=440,
                  title=f"Review helpfulness (record: {df['helpful_vote'].max()} votes)")
fig.show()

## 5. Reviews over time

How the number of reviews grew, how the average rating changed, and whether there is seasonality.
The slider under the average-rating chart lets you zoom into selected years.

In [43]:
per_year = df.groupby(df["timestamp"].dt.year).size().reset_index(name="count")
per_year.columns = ["year", "count"]
fig = px.bar(per_year, x="year", y="count", title="Number of reviews over time",
             labels={"year": "Year", "count": "Number of reviews"})
fig.update_traces(marker_color=NAVY)
fig.show()

In [44]:
yearly = df.groupby(df["timestamp"].dt.year)["rating"].mean().reset_index()
yearly.columns = ["year", "avg_rating"]
fig = px.line(yearly, x="year", y="avg_rating", markers=True,
              title="Average rating over time",
              labels={"year": "Year", "avg_rating": "Average rating"})
fig.update_traces(line_color=PLUM, line_width=3, marker=dict(size=8))
fig.update_yaxes(range=[1, 5])
fig.update_xaxes(rangeslider_visible=True)
fig.show()

In [45]:
tmp = df.assign(year=df["timestamp"].dt.year, month=df["timestamp"].dt.month)
pivot = tmp.pivot_table(index="year", columns="month", values="rating",
                        aggfunc="size", fill_value=0)
fig = px.imshow(pivot, aspect="auto", color_continuous_scale="Viridis",
                title="Review seasonality (year x month)",
                labels=dict(x="Month", y="Year", color="Number of reviews"))
fig.update_xaxes(dtick=1)
fig.show()

## 6. Products and catalog (metadata)

Now we look at **products**, not individual reviews.

> ⚠️ The review sample and the product sample come from different parts of the dataset and
> **barely overlap** (a few dozen shared products). That is why we analyze categories and prices at the
> **product catalog** level (`meta_clean`) rather than on a join with reviews.

In [46]:
per_product = df.groupby("parent_asin").size().reset_index(name="n_reviews")
fig = px.histogram(per_product, x="n_reviews", log_y=True,
                   title="Reviews per product - a long tail",
                   labels={"n_reviews": "Number of reviews per product"})
fig.update_traces(marker_color=AMBER)
fig.update_layout(yaxis_title="Number of products (log)", bargap=0.05)
fig.show()
print(f"Products with exactly 1 review : {(per_product['n_reviews'] == 1).mean():.0%}")
print(f"Most reviews on a single product: {per_product['n_reviews'].max()}")

Products with exactly 1 review : 71%
Most reviews on a single product: 4897


In [47]:
cat = meta.dropna(subset=["main_category"]).copy()
cat["category_l2"] = cat["category_l2"].fillna("(other)")
cat["rating_number"] = cat["rating_number"].fillna(0)
fig = px.treemap(cat, path=[px.Constant("All"), "main_category", "category_l2"],
                 values="rating_number", color="main_category",
                 title="Product categories by total number of ratings")
fig.update_layout(margin=dict(t=60, l=10, r=10, b=10), height=560)
fig.show()

In [48]:
pr = meta.dropna(subset=["price", "average_rating", "rating_number"]).copy()
pr = pr[(pr["price"] > 0) & (pr["price"] <= pr["price"].quantile(0.99))]
fig = px.scatter(pr, x="price", y="average_rating", size="rating_number",
                 color="main_category", hover_name="title",
                 opacity=0.6, size_max=45, log_x=True,
                 title="Price vs. average product rating",
                 labels={"price": "Price (log scale)", "average_rating": "Average rating",
                         "main_category": "Category"})
fig.update_yaxes(range=[0.8, 5.2])
fig.show()

## Takeaways

- **J-shaped ratings** - ~65% of reviews are 5★, which is typical for e-commerce platforms.
- **Short, but with a long tail** - the median review is ~200 characters, yet the longest reach ~10,000;
  negative opinions tend to be longer.
- **Vocabulary of emotion** - negative reviews reveal words like *return, refund, waste, broke*, while
  positive ones show *love, perfect, easy, great*.
- **Helpfulness is a long tail** - most reviews have 0 votes, a few collect hundreds.
- **Over time** - the number of reviews grows over the years, while the average rating stays high and stable.
- **Catalog** - *Computers* and *All Electronics* dominate; price is weakly related to average rating.

This is the starting point for **text embeddings and dimensionality reduction**
(PCA / UMAP / PaCMAP / FIt-SNE), where we look for clusters of similar opinions.